**Imports**

In [23]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

**Code**

Here we made a variable called "df" inside of it we put the .csv

In [5]:
df = pd.read_csv("AmesHousing.csv")

Here we use one-hot encoding on two *categorical* columns, which are : Heating and Central Air. Then putting our outputs into either one of two prefixes depending on where the data came from, and the two prefixes are : Heat and C-Air.

In [10]:
oneHotEncoding = pd.get_dummies(df, columns = ["Heating", "Central Air"], prefix = ["Heat", "C-Air"])
print(oneHotEncoding.head())

   Order        PID  MS SubClass MS Zoning  Lot Frontage  Lot Area Street  \
0      1  526301100           20        RL         141.0     31770   Pave   
1      2  526350040           20        RH          80.0     11622   Pave   
2      3  526351010           20        RL          81.0     14267   Pave   
3      4  526353030           20        RL          93.0     11160   Pave   
4      5  527105010           60        RL          74.0     13830   Pave   

  Alley Lot Shape Land Contour  ... Sale Condition SalePrice Heat_Floor  \
0   NaN       IR1          Lvl  ...         Normal    215000      False   
1   NaN       Reg          Lvl  ...         Normal    105000      False   
2   NaN       IR1          Lvl  ...         Normal    172000      False   
3   NaN       Reg          Lvl  ...         Normal    244000      False   
4   NaN       IR1          Lvl  ...         Normal    189900      False   

  Heat_GasA Heat_GasW Heat_Grav Heat_OthW Heat_Wall  C-Air_N  C-Air_Y  
0      True   

Here we use ordinal encoding to encode every piece of data in the column "Kitchen Qual" into three categories and they are : "low", "medium" and "high" (From course slides)

In [22]:
quality_map = {"Po" : "low",
               "Fa" : "low",
               "TA" : "medium",
               "Gd" : "high",
               "Ex" : "high"} # From ChatGPT, I couldn't fix a problem which was when I ONLY use ordinal encoding everything sets to "-1"

ordinal_enc = pd.CategoricalDtype(categories = ["low", "medium", "high"], ordered = True)

df["Kitchen Qual Level"] = df["Kitchen Qual"].map(quality_map)
df["Kitchen Qual Ordinal"] = df["Kitchen Qual Level"].astype(ordinal_enc).cat.codes

print(df["Kitchen Qual Ordinal"])

0       1
1       1
2       2
3       2
4       1
       ..
2925    1
2926    1
2927    1
2928    1
2929    1
Name: Kitchen Qual Ordinal, Length: 2930, dtype: int8


Here we are using Standard Scaler to scale two columns and they are : "Total Bsmt SF" and "1st Flr SF". There are two outputs and they are for mean and std (Standard Devation) (From course slides)

In [30]:
std_scaler = StandardScaler()
df[["BSMT_std", "FLR_std"]] = std_scaler.fit_transform(df[["Total Bsmt SF", "1st Flr SF"]])
print(df[["BSMT_std", "FLR_std"]].agg(["mean", "std"]))

          BSMT_std       FLR_std
mean -2.255306e-16  1.818795e-17
std   1.000171e+00  1.000171e+00


Here we made 2 different *Domain Driven* features, one of them is *"price_per_sqft"* which was taken from the instruction for this phase and tells you how much every square foot of this house costs, then we have the next feature which is *"overall score"* which was taken from Google Gemini and it tells you the overall score of the house out of 100 by multiplying the condition and the quality of the house (Note : This feature's **idea** was only taken from Google Gemini, the code was written by me) (I have looked and read the course slides to remember how to do it correctly)

In [ ]:
df["price_per_sqft"] = np.where(df["Gr Liv Area"] > 0, df["SalePrice"] / df["Gr Liv Area"], 0)
df["overall_score"] = df["Overall Qual"] * df["Overall Cond"]

print(df[["SalePrice", "Gr Liv Area", "price_per_sqft"]].head())
print("\n")
print(df[["Overall Qual", "Overall Cond", "overall_score"]].head())

   SalePrice  Gr Liv Area  price_per_sqft
0     215000         1656      129.830918
1     105000          896      117.187500
2     172000         1329      129.420617
3     244000         2110      115.639810
4     189900         1629      116.574586


   Overall Qual  Overall Cond  overall_score
0             6             5             30
1             5             6             30
2             6             6             36
3             7             5             35
4             5             5             25


Create 1 interaction feature: multiply two columns that logically go together (e.g., quality × area)

Log-transform 1 skewed column using np.log1p() — show the histogram before and after

Bin 1 column into meaningful groups (e.g., age groups: New, Recent, Old)

Remove redundant features: find highly correlated pairs (r > 0.95) and drop one